**Generative AI Use**: For the purposes of the assignments, the use of generative AI is subject to the same policies regarding collaboration. Just as with other collaborators, each student must write down the solutions independently of the output of the interaction and the submission should include a note denoting the nature of the collaboration. The use of generative AI tools to substantially complete sections of the assignments is not in line with the spirit of the assignments, and would be a violation of the [Honor Code](https://communitystandards.stanford.edu/policies-and-guidance/honor-code).
**关于生成式 AI 的使用**：就本作业而言，使用生成式 AI 与其他形式的协作遵循同样的政策。正如对待其他协作者一样，每位学生都必须独立于交互产生的输出来写下自己的解答，并且提交材料中应注明该协作的性质。使用生成式 AI 工具来实质性地完成作业的各个部分，不符合本作业的精神，将构成对[荣誉准则](https://communitystandards.stanford.edu/policies-and-guidance/honor-code)的违反。


In [ ]:
# This mounts your Google Drive to the Colab VM.
# 这一步把你的 Google Drive 挂载到 Colab 虚拟机上。
from google.colab import drive
drive.mount('/content/drive')

# TODO: Enter the foldername in your Drive where you have saved the unzipped
# assignment folder, e.g. 'cs231n/assignments/assignment1/'
# TODO: 在这里填入你 Drive 中存放已解压作业文件夹的那个目录名，
# 例如 'cs231n/assignments/assignment1/'
FOLDERNAME = 'cs231n/assignments/assignment1/'
assert FOLDERNAME is not None, "[!] Enter the foldername."

# Now that we've mounted your Drive, this ensures that
# the Python interpreter of the Colab VM can load
# python files from within it.
# Drive 挂载好之后，这一步保证 Colab 虚拟机里的 Python 解释器
# 能够从该目录中加载 python 文件。
import sys
sys.path.append('/content/drive/My Drive/{}'.format(FOLDERNAME))

# This downloads the CIFAR-10 dataset to your Drive
# if it doesn't already exist.
# 如果 CIFAR-10 数据集还不存在，这一步会把它下载到你的 Drive 里。
%cd /content/drive/My\ Drive/$FOLDERNAME/cs231n/datasets/
!bash get_datasets.sh
%cd /content/drive/My\ Drive/$FOLDERNAME


# k-Nearest Neighbor (kNN) exercise
# k 近邻（kNN）练习

*Complete and hand in this completed worksheet (including its outputs and any supporting code outside of the worksheet) with your assignment submission. For more details see the [assignments page](http://vision.stanford.edu/teaching/cs231n/assignments.html) on the course website.*
*请把填写完成的这份作业表（包括它的输出，以及作业表之外的任何支撑代码）随作业一并提交。更多细节见课程网站上的[作业页面](http://vision.stanford.edu/teaching/cs231n/assignments.html)。*

The kNN classifier consists of two stages:
kNN 分类器分为两个阶段：

- During training, the classifier takes the training data and simply remembers it
- During testing, kNN classifies every test image by comparing to all training images and transfering the labels of the k most similar training examples
- The value of k is cross-validated

- 训练阶段：分类器接收训练数据，只是把它记住而已
- 测试阶段：kNN 把每张测试图像与所有训练图像进行比较，然后采用最相似的 k 个训练样本的标签作为分类结果
- k 的取值通过交叉验证来确定

In this exercise you will implement these steps and understand the basic Image Classification pipeline, cross-validation, and gain proficiency in writing efficient, vectorized code.
在这个练习中，你将实现上述步骤，理解基本的图像分类流程和交叉验证，并逐步熟练编写高效的向量化代码。


In [ ]:
# Run some setup code for this notebook.
# 运行一些为本 notebook 准备的初始化代码。

import random
import numpy as np
from cs231n.data_utils import load_CIFAR10
import matplotlib.pyplot as plt

# This is a bit of magic to make matplotlib figures appear inline in the notebook
# rather than in a new window.
# 这是一点小魔法，让 matplotlib 的图形直接内嵌显示在 notebook 里，
# 而不是弹出一个新窗口。
%matplotlib inline
plt.rcParams['figure.figsize'] = (10.0, 8.0) # set default size of plots
                                             # 设置图形的默认尺寸
plt.rcParams['image.interpolation'] = 'nearest'
plt.rcParams['image.cmap'] = 'gray'

# Some more magic so that the notebook will reload external python modules;
# see http://stackoverflow.com/questions/1907993/autoreload-of-modules-in-ipython
# 再一点魔法，让 notebook 能够重新加载外部的 python 模块；
# 参见 http://stackoverflow.com/questions/1907993/autoreload-of-modules-in-ipython
%load_ext autoreload
%autoreload 2


In [ ]:
# Load the raw CIFAR-10 data.
# 加载原始的 CIFAR-10 数据。
cifar10_dir = 'cs231n/datasets/cifar-10-batches-py'

# Cleaning up variables to prevent loading data multiple times (which may cause memory issue)
# 清理变量，避免多次重复加载数据（重复加载可能导致内存问题）
try:
   del X_train, y_train
   del X_test, y_test
   print('Clear previously loaded data.')
except:
   pass

X_train, y_train, X_test, y_test = load_CIFAR10(cifar10_dir)

# As a sanity check, we print out the size of the training and test data.
# 作为一次合理性检查，我们打印出训练数据和测试数据的尺寸。
print('Training data shape: ', X_train.shape)
print('Training labels shape: ', y_train.shape)
print('Test data shape: ', X_test.shape)
print('Test labels shape: ', y_test.shape)


In [ ]:
# Visualize some examples from the dataset.
# We show a few examples of training images from each class.
# 可视化数据集中的一些样本。
# 我们从每个类别中展示若干张训练图像作为例子。
classes = ['plane', 'car', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']
num_classes = len(classes)
samples_per_class = 7
for y, cls in enumerate(classes):
    idxs = np.flatnonzero(y_train == y)
    idxs = np.random.choice(idxs, samples_per_class, replace=False)
    for i, idx in enumerate(idxs):
        plt_idx = i * num_classes + y + 1
        plt.subplot(samples_per_class, num_classes, plt_idx)
        plt.imshow(X_train[idx].astype('uint8'))
        plt.axis('off')
        if i == 0:
            plt.title(cls)
plt.show()


In [ ]:
# Subsample the data for more efficient code execution in this exercise
# 对数据做二次采样，让本练习中的代码执行得更快
num_training = 5000
mask = list(range(num_training))
X_train = X_train[mask]
y_train = y_train[mask]

num_test = 500
mask = list(range(num_test))
X_test = X_test[mask]
y_test = y_test[mask]

# Reshape the image data into rows
# 把图像数据重塑成一行一个样本的形式
X_train = np.reshape(X_train, (X_train.shape[0], -1))
X_test = np.reshape(X_test, (X_test.shape[0], -1))
print(X_train.shape, X_test.shape)


In [ ]:
from cs231n.classifiers import KNearestNeighbor

# Create a kNN classifier instance.
# Remember that training a kNN classifier is a noop:
# the Classifier simply remembers the data and does no further processing
# 创建一个 kNN 分类器实例。
# 记住：训练一个 kNN 分类器是个空操作（noop）：
# 分类器只是把数据记住，不做任何进一步的处理
classifier = KNearestNeighbor()
classifier.train(X_train, y_train)


We would now like to classify the test data with the kNN classifier. Recall that we can break down this process into two steps:
现在我们想用 kNN 分类器对测试数据做分类。回忆一下，这个过程可以拆成两步：

1. First we must compute the distances between all test examples and all train examples.
2. Given these distances, for each test example we find the k nearest examples and have them vote for the label

1. 首先，我们要计算所有测试样本与所有训练样本之间的距离。
2. 有了这些距离，对每个测试样本，我们找出 k 个最近的样本，让它们投票决定标签。

Lets begin with computing the distance matrix between all training and test examples. For example, if there are **Ntr** training examples and **Nte** test examples, this stage should result in a **Nte x Ntr** matrix where each element (i,j) is the distance between the i-th test and j-th train example.
我们先从计算所有训练样本与测试样本之间的距离矩阵开始。举例来说，如果有 **Ntr** 个训练样本和 **Nte** 个测试样本，这一步应该得到一个 **Nte x Ntr** 的矩阵，其中每个元素 (i,j) 是第 i 个测试样本与第 j 个训练样本之间的距离。

**Note: For the three distance computations that we require you to implement in this notebook, you may not use the np.linalg.norm() function that numpy provides.**
**注意：本 notebook 要求你实现的三个距离计算，都不允许使用 numpy 提供的 np.linalg.norm() 函数。**

First, open `cs231n/classifiers/k_nearest_neighbor.py` and implement the function `compute_distances_two_loops` that uses a (very inefficient) double loop over all pairs of (test, train) examples and computes the distance matrix one element at a time.
首先打开 `cs231n/classifiers/k_nearest_neighbor.py`，实现函数 `compute_distances_two_loops`。它用（效率很低的）双重循环遍历所有 (测试, 训练) 样本对，一次一个元素地计算出距离矩阵。


In [ ]:
# Open cs231n/classifiers/k_nearest_neighbor.py and implement
# compute_distances_two_loops.
# 打开 cs231n/classifiers/k_nearest_neighbor.py，实现
# compute_distances_two_loops。

# Test your implementation:
# 测试你的实现：
dists = classifier.compute_distances_two_loops(X_test)
print(dists.shape)


In [ ]:
# We can visualize the distance matrix: each row is a single test example and
# its distances to training examples
# 我们可以把距离矩阵可视化：每一行代表一个测试样本，
# 以及它到各个训练样本的距离
plt.imshow(dists, interpolation='none')
plt.show()


**Inline Question 1**
**课堂提问 1**

Notice the structured patterns in the distance matrix, where some rows or columns are visibly brighter. (Note that with the default color scheme black indicates low distances while white indicates high distances.)
注意距离矩阵中那些有规律的图案：某些行或列明显更亮。（注意在默认配色下，黑色表示距离小，白色表示距离大。）

- What in the data is the cause behind the distinctly bright rows?
- What causes the columns?

- 数据中的什么因素造成了那些特别亮的行？
- 那些列又是由什么造成的？

$\color{blue}{\textit Your Answer:}$ *fill this in.*
**你的答案：** *在这里填写。*



In [ ]:
# Now implement the function predict_labels and run the code below:
# We use k = 1 (which is Nearest Neighbor).
# 现在实现函数 predict_labels，然后运行下面的代码：
# 我们取 k = 1（也就是最近邻）。
y_test_pred = classifier.predict_labels(dists, k=1)

# Compute and print the fraction of correctly predicted examples
# 计算并打印预测正确的样本所占的比例
num_correct = np.sum(y_test_pred == y_test)
accuracy = float(num_correct) / num_test
print('Got %d / %d correct => accuracy: %f' % (num_correct, num_test, accuracy))


You should expect to see approximately `27%` accuracy. Now lets try out a larger `k`, say `k = 5`:
你应该会看到大约 `27%` 的准确率。现在我们试一个更大的 `k`，比如 `k = 5`：


In [ ]:
y_test_pred = classifier.predict_labels(dists, k=5)
num_correct = np.sum(y_test_pred == y_test)
accuracy = float(num_correct) / num_test
print('Got %d / %d correct => accuracy: %f' % (num_correct, num_test, accuracy))

You should expect to see a slightly better performance than with `k = 1`.
你应该会看到比 `k = 1` 时略好一点的表现。


**Inline Question 2**
**课堂提问 2**

We can also use other distance metrics such as L1 distance.
我们也可以使用其他的距离度量，比如 L1 距离。
For pixel values $p_{ij}^{(k)}$ at location $(i,j)$ of some image $I_k$,
对于某张图像 $I_k$ 在位置 $(i,j)$ 处的像素值 $p_{ij}^{(k)}$，

the mean $\mu$ across all pixels over all images is $$\mu=\frac{1}{nhw}\sum_{k=1}^n\sum_{i=1}^{h}\sum_{j=1}^{w}p_{ij}^{(k)}$$
所有图像全体像素的均值 $\mu$ 为：
And the pixel-wise mean $\mu_{ij}$ across all images is
而所有图像在第 $(i,j)$ 个像素位置上的逐像素均值 $\mu_{ij}$ 为：
$$\mu_{ij}=\frac{1}{n}\sum_{k=1}^np_{ij}^{(k)}.$$
The general standard deviation $\sigma$ and pixel-wise standard deviation $\sigma_{ij}$ is defined similarly.
一般的标准差 $\sigma$ 和逐像素标准差 $\sigma_{ij}$ 定义与之类似。

Which of the following preprocessing steps will not change the performance of a Nearest Neighbor classifier that uses L1 distance? Select all that apply. To clarify, both training and test examples are preprocessed in the same way.
下面哪些预处理步骤不会改变使用 L1 距离的最近邻分类器的表现？选出所有符合的选项。需要说明的是，训练样本和测试样本施加的是相同的预处理。

1. Subtracting the mean $\mu$ ($\tilde{p}_{ij}^{(k)}=p_{ij}^{(k)}-\mu$.)
2. Subtracting the per pixel mean $\mu_{ij}$  ($\tilde{p}_{ij}^{(k)}=p_{ij}^{(k)}-\mu_{ij}$.)
3. Subtracting the mean $\mu$ and dividing by the standard deviation $\sigma$.
4. Subtracting the pixel-wise mean $\mu_{ij}$ and dividing by the pixel-wise standard deviation $\sigma_{ij}$.
5. Rotating the coordinate axes of the data, which means rotating all the images by the same angle. Empty regions in the image caused by rotation are padded with a same pixel value and no interpolation is performed.

1. 减去均值 $\mu$（$\tilde{p}_{ij}^{(k)}=p_{ij}^{(k)}-\mu$。）
2. 减去逐像素均值 $\mu_{ij}$（$\tilde{p}_{ij}^{(k)}=p_{ij}^{(k)}-\mu_{ij}$。）
3. 减去均值 $\mu$ 并除以标准差 $\sigma$。
4. 减去逐像素均值 $\mu_{ij}$ 并除以逐像素标准差 $\sigma_{ij}$。
5. 旋转数据的坐标轴，也就是把所有图像旋转同一个角度。旋转在图像中造成的空白区域用同一个像素值填充，且不做插值。

$\color{blue}{\textit Your Answer:}$
**你的答案：**


$\color{blue}{\textit Your Explanation:}$
**你的解释：**


In [ ]:
# Now lets speed up distance matrix computation by using partial vectorization
# with one loop. Implement the function compute_distances_one_loop and run the
# code below:
# 现在我们用一层循环的半向量化方式来加速距离矩阵的计算。
# 实现函数 compute_distances_one_loop，然后运行下面的代码：
dists_one = classifier.compute_distances_one_loop(X_test)

# To ensure that our vectorized implementation is correct, we make sure that it
# agrees with the naive implementation. There are many ways to decide whether
# two matrices are similar; one of the simplest is the Frobenius norm. In case
# you haven't seen it before, the Frobenius norm of two matrices is the square
# root of the squared sum of differences of all elements; in other words, reshape
# the matrices into vectors and compute the Euclidean distance between them.
# 为确保我们的向量化实现是正确的，要验证它与朴素实现的结果一致。
# 判断两个矩阵是否相近有很多种方法，其中最简单的一种是 Frobenius 范数。
# 如果你之前没见过：两个矩阵的 Frobenius 范数，就是所有元素差的平方和再开平方根；
# 换句话说，把两个矩阵展平成向量，计算它们之间的欧氏距离。
difference = np.linalg.norm(dists - dists_one, ord='fro')
print('One loop difference was: %f' % (difference, ))
if difference < 0.001:
    print('Good! The distance matrices are the same')
else:
    print('Uh-oh! The distance matrices are different')


In [ ]:
# Now implement the fully vectorized version inside compute_distances_no_loops
# and run the code
# 现在在 compute_distances_no_loops 里实现完全向量化的版本，
# 然后运行代码
dists_two = classifier.compute_distances_no_loops(X_test)

# check that the distance matrix agrees with the one we computed before:
# 检查这个距离矩阵与我们之前算出来的是否一致：
difference = np.linalg.norm(dists - dists_two, ord='fro')
print('No loop difference was: %f' % (difference, ))
if difference < 0.001:
    print('Good! The distance matrices are the same')
else:
    print('Uh-oh! The distance matrices are different')


In [ ]:
# Let's compare how fast the implementations are
# 我们来比较一下这几种实现的快慢
def time_function(f, *args):
    """
    Call a function f with args and return the time (in seconds) that it took to execute.
    调用函数 f 并传入 args，返回它执行所花的时间（单位：秒）。
    """
    import time
    tic = time.time()
    f(*args)
    toc = time.time()
    return toc - tic

two_loop_time = time_function(classifier.compute_distances_two_loops, X_test)
print('Two loop version took %f seconds' % two_loop_time)

one_loop_time = time_function(classifier.compute_distances_one_loop, X_test)
print('One loop version took %f seconds' % one_loop_time)

no_loop_time = time_function(classifier.compute_distances_no_loops, X_test)
print('No loop version took %f seconds' % no_loop_time)

# You should see significantly faster performance with the fully vectorized implementation!
# 你应该会看到，完全向量化的实现要快得多！

# NOTE: depending on what machine you're using,
# you might not see a speedup when you go from two loops to one loop,
# and might even see a slow-down.
# 注意：取决于你用的机器，
# 从两层循环改为一层循环时，你可能看不到加速，
# 甚至可能反而变慢。


### Cross-validation
### 交叉验证

We have implemented the k-Nearest Neighbor classifier but we set the value k = 5 arbitrarily. We will now determine the best value of this hyperparameter with cross-validation.
我们已经实现了 k 近邻分类器，但 k = 5 这个值是随便定的。现在我们要用交叉验证来确定这个超参数的最佳取值。


In [ ]:
num_folds = 5
k_choices = [1, 3, 5, 8, 10, 12, 15, 20, 50, 100]

X_train_folds = []
y_train_folds = []
################################################################################
# TODO:                                                                        #
# Split up the training data into folds. After splitting, X_train_folds and    #
# y_train_folds should each be lists of length num_folds, where                #
# y_train_folds[i] is the label vector for the points in X_train_folds[i].     #
# Hint: Look up the numpy array_split function.                                #
# TODO:                                                                        #
# 把训练数据划分成若干折（fold）。划分之后，X_train_folds 和                 #
# y_train_folds 都应该是长度为 num_folds 的列表，其中                         #
# y_train_folds[i] 是 X_train_folds[i] 中这些点对应的标签向量。                #
# 提示：查一下 numpy 的 array_split 函数。                                     #
################################################################################


# A dictionary holding the accuracies for different values of k that we find
# when running cross-validation. After running cross-validation,
# k_to_accuracies[k] should be a list of length num_folds giving the different
# accuracy values that we found when using that value of k.
# 一个字典，保存交叉验证中不同 k 值下得到的准确率。
# 交叉验证跑完之后，k_to_accuracies[k] 应该是一个长度为 num_folds 的列表，
# 给出使用该 k 值时得到的各个准确率。
k_to_accuracies = {}


################################################################################
# TODO:                                                                        #
# Perform k-fold cross validation to find the best value of k. For each        #
# possible value of k, run the k-nearest-neighbor algorithm num_folds times,   #
# where in each case you use all but one of the folds as training data and the #
# last fold as a validation set. Store the accuracies for all fold and all     #
# values of k in the k_to_accuracies dictionary.                               #
# TODO:                                                                        #
# 执行 k 折交叉验证，找出最佳的 k 值。对于每一个可能的 k 取值，               #
# 把 k 近邻算法运行 num_folds 次，其中每一次都用除某一折之外的所有折          #
# 作为训练数据，用剩下那一折作为验证集。                                       #
# 把所有折、所有 k 值对应的准确率都存进 k_to_accuracies 字典。                 #
################################################################################


# Print out the computed accuracies
# 打印出计算得到的准确率
for k in sorted(k_to_accuracies):
    for accuracy in k_to_accuracies[k]:
        print('k = %d, accuracy = %f' % (k, accuracy))


In [ ]:
# plot the raw observations
# 画出原始的观测点
for k in k_choices:
    accuracies = k_to_accuracies[k]
    plt.scatter([k] * len(accuracies), accuracies)

# plot the trend line with error bars that correspond to standard deviation
# 画出趋势线，误差棒对应标准差
accuracies_mean = np.array([np.mean(v) for k,v in sorted(k_to_accuracies.items())])
accuracies_std = np.array([np.std(v) for k,v in sorted(k_to_accuracies.items())])
plt.errorbar(k_choices, accuracies_mean, yerr=accuracies_std)
plt.title('Cross-validation on k')
plt.xlabel('k')
plt.ylabel('Cross-validation accuracy')
plt.show()


In [ ]:
# Based on the cross-validation results above, choose the best value for k,
# retrain the classifier using all the training data, and test it on the test
# data. You should be able to get above 28% accuracy on the test data.
# 根据上面的交叉验证结果选出最佳的 k 值，
# 用全部训练数据重新训练分类器，然后在测试数据上测试。
# 你应该能在测试数据上取得 28% 以上的准确率。
best_k = 1

classifier = KNearestNeighbor()
classifier.train(X_train, y_train)
y_test_pred = classifier.predict(X_test, k=best_k)

# Compute and display the accuracy
# 计算并显示准确率
num_correct = np.sum(y_test_pred == y_test)
accuracy = float(num_correct) / num_test
print('Got %d / %d correct => accuracy: %f' % (num_correct, num_test, accuracy))


**Inline Question 3**
**课堂提问 3**

Which of the following statements about $k$-Nearest Neighbor ($k$-NN) are true in a classification setting, and for all $k$? Select all that apply.
在分类场景下，下面关于 $k$ 近邻（$k$-NN）的哪些说法对任意 $k$ 都成立？选出所有符合的选项。
1. The decision boundary of the k-NN classifier is linear.
2. The training error of a 1-NN will always be lower than or equal to that of 5-NN.
3. The test error of a 1-NN will always be lower than that of a 5-NN.
4. The time needed to classify a test example with the k-NN classifier grows with the size of the training set.
5. None of the above.

1. k-NN 分类器的决策边界是线性的。
2. 1-NN 的训练误差总是小于或等于 5-NN 的训练误差。
3. 1-NN 的测试误差总是小于 5-NN 的测试误差。
4. 用 k-NN 分类器对一个测试样本分类所需的时间，随训练集规模的增大而增长。
5. 以上说法都不对。

$\color{blue}{\textit Your Answer:}$
**你的答案：**


$\color{blue}{\textit Your Explanation:}$
**你的解释：**

